In [ ]:
#Exploring the probelm in stages from entry to attack.

import sys, time, random
from functools import lru_cache, reduce
from operator import xor
import numpy as np
import matplotlib.pyplot as plt
 
sys.setrecursionlimit(10000)
 
 
# Looking at who wins second stage, the Nim game
def nim_sum(heaps):
    return reduce(xor, heaps, 0)
 
 
def nim_mover_wins(heaps, misere=True, k=None):
    """True if the player about to move in the Nim phase wins with best play."""
    if k is not None:                       # take at most k (normal play):
        return reduce(xor, (h % (k + 1) for h in heaps), 0) != 0  
    if misere and all(h <= 1 for h in heaps):
        return sum(heaps) % 2 == 0          
    return nim_sum(heaps) != 0              # Bouton's theorem
 
 
def nim_mover_wins_brute(heaps, misere=True, k=None):
    """Direct game-tree search for the Nim phase, used only to CHECK the formulas."""
    @lru_cache(None)
    def win(h):
        if sum(h) == 0:                     # previous player took the last stone
            return misere                   # misere: they lost, so I win
        for i, x in enumerate(h):
            for t in range(1, (x if k is None else min(x, k)) + 1):
                g = list(h); g[i] -= t
                if not win(tuple(sorted(g))):
                    return True
        return False
    return win(tuple(sorted(heaps)))
 
 
# This is the game which includes Phase 1 and 2
@lru_cache(None)
def build_wins(heaps, r, misere=True, k=None):
    """heaps: sorted tuple, r: stones left to place.
    True if the player about to move (place, or start Nim if r == 0) wins."""
    if r == 0:
        return nim_mover_wins(heaps, misere, k)
    for i in range(len(heaps)):
        if i and heaps[i] == heaps[i - 1]:
            continue                        # equal heaps give the same position
        g = list(heaps); g[i] += 1
        if not build_wins(tuple(sorted(g)), r - 1, misere, k):
            return True
    return False
 
 
def p1_wins(n, s, misere=True, k=None):
    return build_wins((0,) * n, s, misere, k)
 
 
def winner_table(N, S, misere=True, k=None):
    return np.array([[1 if p1_wins(n, s, misere, k) else 2 for s in range(1, S + 1)]
                     for n in range(1, N + 1)])
 
 
# This is simulation steps
def simulate(n, s, p1_builder, p2_builder, misere=True, rng=random):
    """Play the building phase with the given strategies, then Nim optimally.
    Returns the winner (1 or 2)."""
    heaps = [0] * n
    for t in range(s):
        builder = p1_builder if t % 2 == 0 else p2_builder
        heaps[builder(heaps, s - t, rng)] += 1
    nim_mover = 1 if s % 2 == 0 else 2      # turns keep alternating
    return nim_mover if nim_mover_wins(heaps, misere) else 3 - nim_mover
 
 
def random_builder(heaps, r, rng):
    return rng.randrange(len(heaps))
 
 
def pairing_builder(heaps, r, rng):
    """Player 2's strategy for even n: answer in the partner heap of P1's last stone."""
    n = len(heaps)
    # the partner pair that is unbalanced is the one P1 just played in
    for i in range(0, n - 1, 2):
        if heaps[i] != heaps[i + 1]:
            return i if heaps[i] < heaps[i + 1] else i + 1
    return rng.randrange(n)
 
 
def optimal_builder(misere=True):
    def b(heaps, r, rng):
        for i in range(len(heaps)):
            g = list(heaps); g[i] += 1
            if not build_wins(tuple(sorted(g)), r - 1, misere):
                return i
        return rng.randrange(len(heaps))
    return b
 
 
def zero_triples(s):
    """Final 3-heap positions (a, b, c), a+b+c = s, with nim sum 0."""
    return [(a, b, s - a - b) for a in range(s + 1) for b in range(s + 1 - a)
            if a ^ b ^ (s - a - b) == 0]
 
 
if __name__ == "__main__":
    # Doing our first check with Nim formulas against brute force
    import itertools
    for heaps in itertools.product(range(5), repeat=3):
        for mis in (True, False):
            assert nim_mover_wins(heaps, mis) == nim_mover_wins_brute(heaps, mis)
        for k in (1, 2, 3):
            assert nim_mover_wins(heaps, False, k) == nim_mover_wins_brute(heaps, False, k)
    print("Nim-phase formulas agree with brute force on all heaps in {0..4}^3")
 
    # Tables (Section 2)
    for mis in (True, False):
        print("\nmisere" if mis else "\nnormal play", "(rows n = 1..8, columns s = 1..30)")
        for n, row in enumerate(winner_table(8, 30, mis), 1):
            print(n, "".join(map(str, row)))
 
    # Conjectures
    print("\nn = 3, normal play, even s where P2 wins:",
          [s for s in range(2, 131, 2) if not p1_wins(3, s, False)])
    for n in range(5, 14, 2):
        print(f"n = {n}, normal play, even s where P2 wins:",
              [s for s in range(2, 41, 2) if not p1_wins(n, s, False)])
    print("misere differs from normal play at (n, s):",
          [(n, s) for n in range(1, 9) for s in range(1, 31)
           if p1_wins(n, s, True) != p1_wins(n, s, False)])
    print("zero-nim-sum targets for n = 3:",
          {s: len(zero_triples(s)) for s in range(2, 31, 2)})
 
    # Variant, when we take at most k
    for k in (1, 2, 3):
        print(f"\nnormal play, take at most {k}:")
        for n, row in enumerate(winner_table(6, 30, False, k), 1):
            print(n, "".join(map(str, row)))
 
    # Looking at timings and a memoised solver
    build_wins.cache_clear()
    t0 = time.time(); p1_wins(7, 30, True); t1 = time.time()
    print(f"\nsolve (n, s) = (7, 30): {t1 - t0:.2f}s, "
          f"{build_wins.cache_info().currsize} positions stored")
 
    # Further simulations
    rng = random.Random(3)
    T = 4000
    print("\nrandom vs random (misere), P1 win rate:")
    for n in (2, 3, 5):
        print(n, [round(sum(simulate(n, s, random_builder, random_builder, True, rng) == 1
                                for _ in range(T)) / T, 2) for s in (4, 8, 12, 16, 20)])
    wins = sum(simulate(4, 20, random_builder, pairing_builder, False, rng) == 2
               for _ in range(T))
    print(f"pairing P2 vs random P1, n=4, s=20, normal play: P2 wins {wins}/{T}")
    wins = sum(simulate(3, 20, optimal_builder(False), random_builder, False, rng) == 1
               for _ in range(T))
    print(f"optimal P1 vs random P2, n=3, s=20, normal play: P1 wins {wins}/{T}")
 
    # Images and Figures
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(["#3b6fb6", "#e0a030"])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
    for ax, mis, title in zip(axes, (True, False), ("misère (last stone loses)",
                                                    "normal play (last stone wins)")):
        W = winner_table(10, 30, mis)
        ax.imshow(W, cmap=cmap, vmin=1, vmax=2, aspect="auto",
                  extent=(0.5, 30.5, 10.5, 0.5))
        ax.set_title(title); ax.set_xlabel("stones $s$")
        ax.set_xticks(range(2, 31, 4)); ax.set_yticks(range(1, 11))
    axes[0].set_ylabel("heaps $n$")
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(color="#3b6fb6", label="Player 1 wins"),
                        Patch(color="#e0a030", label="Player 2 wins")],
               loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04), frameon=False)
    fig.tight_layout(); fig.savefig("fig_winners.png", dpi=200, bbox_inches="tight")
 
    fig, ax = plt.subplots(figsize=(6, 3))
    S = list(range(2, 131, 2))
    ax.semilogy(S, [len(zero_triples(s)) for s in S], ".", color="grey",
                label="zero-nim-sum final positions")
    P2 = [s for s in S if not p1_wins(3, s, False)]
    ax.semilogy(P2, [len(zero_triples(s)) for s in P2], "o", color="#e0a030",
                label="P2 wins ($s = 2^m-2$)")
    ax.set_xlabel("stones $s$ (even)"); ax.set_ylabel("count")
    ax.legend(frameon=False); fig.tight_layout()
    fig.savefig("fig_n3.png", dpi=200, bbox_inches="tight")
    print("\nfigures saved")